In [ ]:
!pip install -q ultralytics

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO
from collections import Counter

In [ ]:
MODEL_PATH = "/kaggle/input/models/tanya24jain/indian-smart-traffic-yolov8-model/pytorch/v1/1/best (2).pt"

VIDEO_DIR = "/kaggle/input/datasets/tanya24jain/real-world-unseen-traffic-video-dataset1/unseen_test_videos"

In [ ]:
videos = sorted(os.listdir(VIDEO_DIR))

print("Available Videos:\n")

for i, v in enumerate(videos):
    print(f"{i} - {v}")

In [ ]:
VIDEO_INDEX = 0

VIDEO_PATH = os.path.join(
    VIDEO_DIR,
    videos[VIDEO_INDEX]
)

print(VIDEO_PATH)

In [ ]:
model = YOLO(MODEL_PATH)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)

frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS :", fps)
print("Frames :", frames)
print("Resolution :", width, "x", height)

cap.release()

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    20,
    dtype=int
)

fig, axes = plt.subplots(
    5,
    4,
    figsize=(12,15)
)

for ax, frame_no in zip(
    axes.flatten(),
    frame_numbers
):

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_no
    )

    ret, frame = cap.read()

    if not ret:
        ax.axis("off")
        continue

    ax.imshow(
        cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )
    )

    ax.set_title(
        f"Original {frame_no}",
        fontsize=8
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

cap.release()

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    20,
    dtype=int
)

fig, axes = plt.subplots(6, 3,figsize=(12,15)
)

for ax, frame_no in zip(
    axes.flatten(),
    frame_numbers
):

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_no
    )

    ret, frame = cap.read()

    if not ret:
        ax.axis("off")
        continue

    results = model(
        frame,
        verbose=False
    )

    annotated = results[0].plot()

    ax.imshow(
        cv2.cvtColor(
            annotated,
            cv2.COLOR_BGR2RGB
        )
    )

    ax.set_title(
        f"Frame {frame_no}",
        fontsize=8
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

cap.release()

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    20,
    dtype=int
)

for frame_no in frame_numbers:

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_no
    )

    ret, frame = cap.read()

    if not ret:
        continue

    results = model(
        frame,
        verbose=False
    )

    boxes = results[0].boxes

    counts = Counter()

    for box in boxes:

        cls_id = int(box.cls[0])

        cls_name = model.names[cls_id]

        counts[cls_name] += 1

    print("\n" + "="*40)

    print(
        f"Frame {frame_no}"
    )

    print(
        f"Total Vehicles: {len(boxes)}"
    )

    if len(boxes) == 0:
        print(
            "No vehicles detected"
        )

    for cls, count in sorted(
        counts.items()
    ):
        print(
            f"{cls}: {count}"
        )

cap.release()

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    20,
    dtype=int
)

overall = Counter()

for frame_no in frame_numbers:

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_no
    )

    ret, frame = cap.read()

    if not ret:
        continue

    results = model(
        frame,
        verbose=False
    )

    for box in results[0].boxes:

        cls_id = int(box.cls[0])

        cls_name = model.names[cls_id]

        overall[cls_name] += 1

cap.release()

print("\nOVERALL DETECTIONS")

for cls, count in sorted(
    overall.items()
):
    print(
        f"{cls}: {count}"
    )

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import cv2

model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)

unique_ids = defaultdict(set)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model.track(
        frame,
        persist=True,
        verbose=False
    )

    if results[0].boxes.id is None:
        continue

    track_ids = results[0].boxes.id.cpu().numpy().astype(int)

    class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

    for track_id, class_id in zip(track_ids, class_ids):

        class_name = model.names[class_id]

        unique_ids[class_name].add(track_id)

cap.release()


total = 0

for cls in sorted(unique_ids.keys()):

    count = len(unique_ids[cls])

    total += count

    print(f"{cls}: {count}")


print("Total Video Vehicles:", total)

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)

ret, frame = cap.read()

results = model.track(
    frame,
    persist=True,
    verbose=False
)

annotated = results[0].plot()

plt.figure(figsize=(10,6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Tracked Vehicles with IDs")
plt.show()

cap.release()

In [ ]:
import time
model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_path = "/kaggle/working/final_demo.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

frame_count = 0

start_time = time.time()

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, verbose=False)

    annotated = results[0].plot()

    frame_count += 1

    elapsed = time.time() - start_time

    current_fps = frame_count / elapsed

    cv2.putText(
        annotated,
        f"FPS: {current_fps:.2f}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )

    cv2.putText(
        annotated,
        f"Vehicles: {len(results[0].boxes)}",
        (20,80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )

    out.write(annotated)

cap.release()
out.release()

print("Saved:", output_path)     

In [ ]:
from IPython.display import Video

Video("/kaggle/working/final_demo.mp4", embed=True)

In [ ]:
import os

video_path = "/kaggle/working/final_demo.mp4"

print("Exists:", os.path.exists(video_path))

if os.path.exists(video_path):
    print("Size (MB):", round(os.path.getsize(video_path)/(1024*1024),2))

In [ ]:
import cv2

cap = cv2.VideoCapture("/kaggle/working/final_demo.mp4")

print("Opened:", cap.isOpened())
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))

ret, frame = cap.read()

print("Read Success:", ret)

cap.release()

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture("/kaggle/working/final_demo.mp4")

ret, frame = cap.read()

print("Read:", ret)

if ret:
    plt.figure(figsize=(12,6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

cap.release()

In [ ]:
import os

print(os.path.getsize("/kaggle/working/final_demo.mp4")/1024/1024, "MB")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

video = "/kaggle/working/final_demo.mp4"

cap = cv2.VideoCapture(video)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frames_to_show = np.linspace(
    0,
    total_frames - 1,
    12,
    dtype=int
)

fig, axes = plt.subplots(4, 3, figsize=(15, 12))

for ax, frame_no in zip(axes.flatten(), frames_to_show):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)

    ret, frame = cap.read()

    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {frame_no}")
        ax.axis("off")

plt.tight_layout()
plt.show()

cap.release()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    12,
    dtype=int
)

fig, axes = plt.subplots(4, 3, figsize=(18, 14))

for ax, frame_no in zip(axes.flatten(), frame_numbers):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)

    ret, frame = cap.read()

    if not ret:
        continue

    results = model(frame, verbose=False)

    annotated = results[0].plot()

    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))

    vehicle_count = len(results[0].boxes)

    labels = []

    for box in results[0].boxes:

        cls_id = int(box.cls[0])

        cls_name = model.names[cls_id]

        conf = float(box.conf[0])

        labels.append(f"{cls_name}({conf:.2f})")

    ax.set_title(
        f"Frame {frame_no}\n"
        f"Vehicles: {vehicle_count}\n"
        + ", ".join(labels[:4]),  # max 4 labels
        fontsize=8
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

cap.release()

In [ ]:
VIDEO_INDEX = 1

VIDEO_PATH = os.path.join(
    VIDEO_DIR,
    videos[VIDEO_INDEX]
)

print("Selected Video:")
print(VIDEO_PATH)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)

frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frames / fps

print("FPS :", fps)
print("Frames :", frames)
print("Resolution :", width, "x", height)
print("Duration :", round(duration,2), "sec")

cap.release()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    12,
    dtype=int
)

fig, axes = plt.subplots(4,3,figsize=(15,12))

for ax, frame_no in zip(axes.flatten(), frame_numbers):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)

    ret, frame = cap.read()

    if ret:

        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        ax.set_title(f"Frame {frame_no}")

        ax.axis("off")

plt.tight_layout()
plt.show()

cap.release()

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    12,
    dtype=int
)

fig, axes = plt.subplots(4,3,figsize=(18,14))

for ax, frame_no in zip(axes.flatten(), frame_numbers):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)

    ret, frame = cap.read()

    if not ret:
        continue

    results = model(frame, verbose=False)

    annotated = results[0].plot()

    ax.imshow(
        cv2.cvtColor(
            annotated,
            cv2.COLOR_BGR2RGB
        )
    )

    vehicle_count = len(results[0].boxes)

    labels = []

    for box in results[0].boxes:

        cls_id = int(box.cls[0])

        cls_name = model.names[cls_id]

        conf = float(box.conf[0])

        labels.append(
            f"{cls_name}({conf:.2f})"
        )

    ax.set_title(
        f"Frame {frame_no}\n"
        f"Vehicles:{vehicle_count}\n"
        + ", ".join(labels[:3]),
        fontsize=8
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

cap.release()

In [ ]:
from collections import defaultdict

cap = cv2.VideoCapture(VIDEO_PATH)

unique_ids = defaultdict(set)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model.track(
        frame,
        persist=True,
        verbose=False
    )

    if results[0].boxes.id is None:
        continue

    track_ids = results[0].boxes.id.cpu().numpy().astype(int)

    class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

    for track_id, class_id in zip(track_ids, class_ids):

        class_name = model.names[class_id]

        unique_ids[class_name].add(track_id)

cap.release()

total = 0

print("\nVEHICLE COUNT")

for cls in sorted(unique_ids.keys()):

    count = len(unique_ids[cls])

    total += count

    print(f"{cls}: {count}")

print("-"*30)
print("Total Video Vehicles:", total)

In [ ]:
VIDEO_INDEX = 2
VIDEO_PATH = os.path.join(
    VIDEO_DIR,
    videos[VIDEO_INDEX]
)

print("Selected Video:")
print(VIDEO_PATH)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)

frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frames / fps

print("FPS :", fps)
print("Frames :", frames)
print("Resolution :", width, "x", height)
print("Duration :", round(duration,2), "sec")

cap.release()

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    12,
    dtype=int
)

fig, axes = plt.subplots(4,3,figsize=(15,12))

for ax, frame_no in zip(axes.flatten(), frame_numbers):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)

    ret, frame = cap.read()

    if ret:

        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        ax.set_title(f"Frame {frame_no}")

        ax.axis("off")

plt.tight_layout()
plt.show()

cap.release()

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_numbers = np.linspace(
    0,
    total_frames - 1,
    12,
    dtype=int
)

fig, axes = plt.subplots(4,3,figsize=(18,14))

for ax, frame_no in zip(axes.flatten(), frame_numbers):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)

    ret, frame = cap.read()

    if not ret:
        continue

    results = model(frame, verbose=False)

    annotated = results[0].plot()

    ax.imshow(
        cv2.cvtColor(
            annotated,
            cv2.COLOR_BGR2RGB
        )
    )

    vehicle_count = len(results[0].boxes)

    labels = []

    for box in results[0].boxes:

        cls_id = int(box.cls[0])

        cls_name = model.names[cls_id]

        conf = float(box.conf[0])

        labels.append(
            f"{cls_name}({conf:.2f})"
        )

    ax.set_title(
        f"Frame {frame_no}\n"
        f"Vehicles:{vehicle_count}\n"
        + ", ".join(labels[:3]),
        fontsize=8
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

cap.release()

In [ ]:
from collections import defaultdict

cap = cv2.VideoCapture(VIDEO_PATH)

unique_ids = defaultdict(set)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model.track(
        frame,
        persist=True,
        verbose=False
    )

    if results[0].boxes.id is None:
        continue

    track_ids = results[0].boxes.id.cpu().numpy().astype(int)

    class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

    for track_id, class_id in zip(track_ids, class_ids):

        class_name = model.names[class_id]

        unique_ids[class_name].add(track_id)

cap.release()

total = 0

print("\n VEHICLE COUNT")

for cls in sorted(unique_ids.keys()):

    count = len(unique_ids[cls])

    total += count

    print(f"{cls}: {count}")

print("-"*30)
print("Total Video Vehicles:", total)

In [ ]:
START_INDEX = 3
END_INDEX = 14

In [ ]:
for VIDEO_INDEX in range(START_INDEX, END_INDEX + 1):

    VIDEO_PATH = os.path.join(
        VIDEO_DIR,
        videos[VIDEO_INDEX]
    )

    print("\n" + "-"*60)

    print(f"VIDEO INDEX : {VIDEO_INDEX}")
    print(f"VIDEO NAME  : {videos[VIDEO_INDEX]}")

    cap = cv2.VideoCapture(VIDEO_PATH)

    fps = cap.get(cv2.CAP_PROP_FPS)

    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    duration = frames / fps

    print("FPS :", round(fps,2))
    print("Frames :", frames)
    print("Resolution :", width, "x", height)
    print("Duration :", round(duration,2), "sec")

    cap.release()

In [ ]:
for VIDEO_INDEX in range(START_INDEX, END_INDEX + 1):

    VIDEO_PATH = os.path.join(
        VIDEO_DIR,
        videos[VIDEO_INDEX]
    )

    cap = cv2.VideoCapture(VIDEO_PATH)

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    frame_numbers = np.linspace(
        0,
        total_frames - 1,
        6,
        dtype=int
    )

    fig, axes = plt.subplots(
        2, 3,
        figsize=(12, 6)
    )

    fig.suptitle(
        f"VIDEO {VIDEO_INDEX} : {videos[VIDEO_INDEX]}",
        fontsize=14
    )

    for ax, frame_no in zip(
        axes.flatten(),
        frame_numbers
    ):

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_no
        )

        ret, frame = cap.read()

        if ret:

            ax.imshow(
                cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB
                )
            )

            ax.set_title(
                f"Frame {frame_no}"
            )

            ax.axis("off")

    plt.tight_layout()

    plt.show()

    cap.release()

In [ ]:
for VIDEO_INDEX in range(START_INDEX, END_INDEX + 1):

    VIDEO_PATH = os.path.join(
        VIDEO_DIR,
        videos[VIDEO_INDEX]
    )

    cap = cv2.VideoCapture(VIDEO_PATH)

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    frame_numbers = np.linspace(
        0,
        total_frames - 1,
        6,
        dtype=int
    )

    fig, axes = plt.subplots(
        2, 3,
        figsize=(14, 8)
    )

    fig.suptitle(
        f"DETECTIONS : VIDEO {VIDEO_INDEX}",
        fontsize=14
    )

    for ax, frame_no in zip(
        axes.flatten(),
        frame_numbers
    ):

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_no
        )

        ret, frame = cap.read()

        if not ret:
            continue

        results = model(
            frame,
            verbose=False
        )

        annotated = results[0].plot()

        ax.imshow(
            cv2.cvtColor(
                annotated,
                cv2.COLOR_BGR2RGB
            )
        )

        ax.set_title(
            f"Vehicles: {len(results[0].boxes)}"
        )

        ax.axis("off")

    plt.tight_layout()

    plt.show()

    cap.release()

In [ ]:
from collections import Counter

for VIDEO_INDEX in range(START_INDEX, END_INDEX + 1):

    VIDEO_PATH = os.path.join(
        VIDEO_DIR,
        videos[VIDEO_INDEX]
    )

    print("\n" + "-"*60)
    print(f"VIDEO INDEX : {VIDEO_INDEX}")
    print(f"VIDEO NAME  : {videos[VIDEO_INDEX]}")
    print("-"*60)

    cap = cv2.VideoCapture(VIDEO_PATH)

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    frame_numbers = np.linspace(0,total_frames - 1,6,dtype=int
    )

    for frame_no in frame_numbers:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_no
        )

        ret, frame = cap.read()

        if not ret:
            continue

        results = model(
            frame,
            verbose=False
        )

        counts = Counter()

        for box in results[0].boxes:

            cls_id = int(box.cls[0])

            cls_name = model.names[cls_id]

            counts[cls_name] += 1

        total = sum(counts.values())

        print(f"\nFrame {frame_no}")
        print(f"Total Vehicles: {total}")

        for cls, count in counts.items():
            print(f"{cls}: {count}")

    cap.release()

In [ ]:
from collections import defaultdict

for VIDEO_INDEX in range(START_INDEX, END_INDEX + 1):

    VIDEO_PATH = os.path.join(
        VIDEO_DIR,
        videos[VIDEO_INDEX]
    )

    cap = cv2.VideoCapture(VIDEO_PATH)

    unique_ids = defaultdict(set)

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = model.track(
            frame,
            persist=True,
            verbose=False
        )

        if results[0].boxes.id is None:
            continue

        track_ids = results[0].boxes.id.cpu().numpy().astype(int)

        class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

        for track_id, class_id in zip(
            track_ids,
            class_ids
        ):

            class_name = model.names[class_id]

            unique_ids[class_name].add(track_id)

    cap.release()

    print("\n" + "-"*60)
    print(f"VIDEO INDEX : {VIDEO_INDEX}")
    print(f"VIDEO NAME  : {videos[VIDEO_INDEX]}")
    print("-"*60)

    total = 0

    for cls in sorted(unique_ids.keys()):

        count = len(unique_ids[cls])

        total += count

        print(f"{cls}: {count}")

    print("-"*40)
    print("TOTAL VEHICLES:", total)

In [ ]:
import pandas as pd
from collections import defaultdict
import cv2

summary = []

for VIDEO_INDEX in range(START_INDEX, END_INDEX + 1):

    VIDEO_PATH = os.path.join(VIDEO_DIR, videos[VIDEO_INDEX])

    cap = cv2.VideoCapture(VIDEO_PATH)

    unique_ids = defaultdict(set)

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = model.track(
            frame,
            persist=True,
            verbose=False
        )

        if results[0].boxes.id is None:
            continue

        track_ids = results[0].boxes.id.cpu().numpy().astype(int)

        class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

        for track_id, class_id in zip(track_ids, class_ids):

            class_name = model.names[class_id]

            unique_ids[class_name].add(track_id)

    cap.release()

    truck = len(unique_ids.get("truck", set()))
    auto = len(unique_ids.get("autorickshaw", set()))
    car = len(unique_ids.get("car", set()))
    bike = len(unique_ids.get("bike", set()))
    bus = len(unique_ids.get("bus", set()))


    total = truck + auto + car + bike + bus

    summary.append([
        VIDEO_INDEX,
        videos[VIDEO_INDEX],
        truck,
        auto,
        car,
        bike,
        bus,
        total
    ])

df = pd.DataFrame(
    summary,
    columns=[
        "Video Index",
        "Video Name",
        "Truck",
        "Autorickshaw",
        "car",
        "bike",
        "bus",
        "Total Vehicles"
    ]
)

df                 

In [ ]:
import matplotlib.pyplot as plt

vehicle_totals = {
    "Car": df["car"].sum(),
    "Bike": df["bike"].sum(),
    "Truck": df["Truck"].sum(),
    "Autorickshaw": df["Autorickshaw"].sum(),
    "Bus": df["bus"].sum()
}

plt.figure(figsize=(8,5))

plt.bar(
    vehicle_totals.keys(),
    vehicle_totals.values()
)

plt.title("Overall Vehicle Distribution")

plt.ylabel("Vehicle Count")

plt.xlabel("Vehicle Type")

plt.grid(axis="y")

plt.show()

print(vehicle_totals)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(
    df["Video Index"],
    df["Total Vehicles"],
    marker="o"
)

plt.title("Traffic Count Per Video")

plt.xlabel("Video Index")

plt.ylabel("Total Vehicles")

plt.grid(True)

plt.show()

In [ ]:
max_row = df.loc[
    df["Total Vehicles"].idxmax()
]

print("MOST CROWDED VIDEO")
print("="*50)

print("Video Index :", max_row["Video Index"])
print("Video Name  :", max_row["Video Name"])

print("\nVehicle Counts")
print("Truck        :", max_row["Truck"])
print("Autorickshaw :", max_row["Autorickshaw"])
print("Car          :", max_row["car"])
print("Bike         :", max_row["bike"])
print("Bus          :", max_row["bus"])

print("\nTotal Vehicles :", max_row["Total Vehicles"])